In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
ss = SparkSession.builder.appName('Project').getOrCreate()

In [ ]:
forest = ss.read.csv('/content/ForestFireDataSet.csv',header = True)

In [ ]:
forest.show()

+-----------+-----------+-------------+----------------------------+----------------------+---------------+------------------------+-----------+----+---------------+----+--------+
|   Latitude|  Longitude|Date_mmddyyyy|FFMC_Fine_Fuel_Moisture_Code|DMC_Duff_Moisture_Code|DC_Drought_Code|ISI_Initial_Spread_Index|Temperature|  RH|Wind_Speed_kmph|Rain|Area_ha |
+-----------+-----------+-------------+----------------------------+----------------------+---------------+------------------------+-----------+----+---------------+----+--------+
|  28.628189| 117.117138|   01-01-2018|                       82.98|                281.39|         118.05|                   26.32|      30.41|  57|           2.22|1.32|     244|
|-33.7363181| 25.3908518|    6/23/2019|                       32.13|                290.35|          25.66|                   29.94|       2.74|  65|           6.83|3.16|     997|
| -7.0258984|108.5567057|    7/25/2019|                       35.75|                180.49|         

In [ ]:
forest.columns

['Latitude',
 'Longitude',
 'Date_mmddyyyy',
 'FFMC_Fine_Fuel_Moisture_Code',
 'DMC_Duff_Moisture_Code',
 'DC_Drought_Code',
 'ISI_Initial_Spread_Index',
 'Temperature',
 'RH',
 'Wind_Speed_kmph',
 'Rain',
 'Area_ha ']

In [ ]:
forest.printSchema()

root
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- Date_mmddyyyy: string (nullable = true)
 |-- FFMC_Fine_Fuel_Moisture_Code: string (nullable = true)
 |-- DMC_Duff_Moisture_Code: string (nullable = true)
 |-- DC_Drought_Code: string (nullable = true)
 |-- ISI_Initial_Spread_Index: string (nullable = true)
 |-- Temperature: string (nullable = true)
 |-- RH: string (nullable = true)
 |-- Wind_Speed_kmph: string (nullable = true)
 |-- Rain: string (nullable = true)
 |-- Area_ha : string (nullable = true)



In [ ]:
# forest = forest.withColumnRenamed('Latitude',forest['Latitude'].cast('float'))\
# .withColumnRenamed('Longitude',forest['Longitude'].cast('float'))\
# .withColumnRenamed('FFMC_Fine_Fuel_Moisture_Code',forest['FFMC_Fine_Fuel_Moisture_Code'].cast('float'))\
# .withColumnRenamed('DMC_Duff_Moisture_Code',forest['DMC_Duff_Moisture_Code'].cast('float'))\
# .withColumnRenamed('DC_Drought_Code',forest['DC_Drought_Code'].cast('float'))\
# .withColumnRenamed('ISI_Initial_Spread_Index',forest['ISI_Initial_Spread_Index'].cast('float'))\
# .withColumnRenamed('Temperature',forest['Temperature'].cast('float'))\
# .withColumnRenamed('RH',forest['RH'].cast('float'))\
# .withColumnRenamed('Wind_Speed_kmph',forest['Wind_Speed_kmph'].cast('float'))\
# .withColumnRenamed('Rain',forest['Rain'].cast('float'))

In [ ]:
forest = forest.withColumn('Latitude',forest.Latitude.astype('float'))\
.withColumn('Longitude',forest.Longitude.astype('float'))\
.withColumn('FFMC_Fine_Fuel_Moisture_Code',forest.FFMC_Fine_Fuel_Moisture_Code.astype('float'))\
.withColumn('DMC_Duff_Moisture_Code',forest.DMC_Duff_Moisture_Code.astype('float'))\
.withColumn('DC_Drought_Code',forest.DC_Drought_Code.astype('float'))\
.withColumn('ISI_Initial_Spread_Index',forest.ISI_Initial_Spread_Index.astype('float'))\
.withColumn('Temperature',forest.Temperature.astype('float'))\
.withColumn('RH',forest.RH.astype('float'))\
.withColumn('Wind_Speed_kmph',forest.Wind_Speed_kmph.astype('float'))\
.withColumn('Rain',forest.Rain.astype('float'))

In [ ]:
forest = forest.dropna()

In [ ]:
from pyspark.ml.feature import VectorAssembler

In [ ]:
va = VectorAssembler(
    inputCols = ['Latitude', 'Longitude', 'FFMC_Fine_Fuel_Moisture_Code', 'DMC_Duff_Moisture_Code', 'DC_Drought_Code', 'ISI_Initial_Spread_Index', 'Temperature', 'RH', 'Wind_Speed_kmph', 'Rain'],
    outputCol = 'features'
)

In [ ]:
forest_transformed = va.transform(forest)

In [ ]:
forest_transformed.show()

+----------+----------+-------------+----------------------------+----------------------+---------------+------------------------+-----------+----+---------------+----+--------+--------------------+
|  Latitude| Longitude|Date_mmddyyyy|FFMC_Fine_Fuel_Moisture_Code|DMC_Duff_Moisture_Code|DC_Drought_Code|ISI_Initial_Spread_Index|Temperature|  RH|Wind_Speed_kmph|Rain|Area_ha |            features|
+----------+----------+-------------+----------------------------+----------------------+---------------+------------------------+-----------+----+---------------+----+--------+--------------------+
|  28.62819| 117.11714|   01-01-2018|                       82.98|                281.39|         118.05|                   26.32|      30.41|57.0|           2.22|1.32|     244|[28.6281890869140...|
|-33.736317| 25.390852|    6/23/2019|                       32.13|                290.35|          25.66|                   29.94|       2.74|65.0|           6.83|3.16|     997|[-33.736316680908...|
|-7.0

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [ ]:
kmeans = KMeans().setK(3).setSeed(1)

In [ ]:
model = kmeans.fit(forest_transformed)

In [ ]:
pred = model.transform(forest_transformed)

In [ ]:
pred.show()

In [ ]:
pred_out = pred

In [ ]:
pred_out = pred_out.drop('features')

In [ ]:
pred_out.write.csv('forest')

Now we can use this file for tableau dashboard creations.